# Напишите заголовок проекта здесь

- Автор: Федоров Алексей Анатольевич
- Дата: 12.04.2026

### Цели и задачи проекта

Найти главные факторы, влияющие на увеличение рейтинга точки питания в Москве: расположение, размер, режим работы, тип точки, ценовой сегмент. Чтобы дать рекомендации инвесторам Shut Up and Take My Money, как открывать новые точки общественного питания с более высокой проходимостью, так как рейтинг увеличивает проходимость, а значит может увеличить выручку.

### Описание данных

#### Датасет **rest_info.csv** - информация о точках общественного питания
* **name** — название заведения;
* **address** — адрес заведения;
* **district** — административный район (например, Центральный административный округ);
* **category** — категория заведения (например, «кафе», «пиццерия» или «кофейня»);
* **hours** — информация о днях и часах работы;
* **rating** — рейтинг заведения по оценкам пользователей в Яндекс Картах (макс. 5.0);
* **chain** — является ли заведение сетевым (0 — нет, 1 — да);
* **seats** — количество посадочных мест.

#### Датасет **rest_price.csv** - информация о среднем чеке в точках питания
* **price** — категория цен в заведении, например «средние», «ниже среднего», «выше среднего» и так далее;

* **avg_bill** — строка, которая хранит среднюю стоимость заказа в виде диапазона, например:
    * «Средний счёт: 1000–1500 ₽»;
    * «Цена чашки капучино: 130–220 ₽»;
    * «Цена бокала пива: 400–600 ₽» и так далее;

* **middle_avg_bill** — число с оценкой среднего чека, которое указано только для значений из столбца **avg_bill**, начинающихся с подстроки «Средний счёт»:
    * Если в строке указан ценовой диапазон из двух значений, в столбец войдёт медиана этих двух значений.
    * Если в строке указано одно число — цена без диапазона, то в столбец войдёт это число.
    * Если значения нет или оно не начинается с подстроки «Средний счёт», то в столбец ничего не войдёт.

* **middle_coffee_cup** — число с оценкой одной чашки капучино, которое указано только для значений из столбца **avg_bill**, начинающихся с подстроки «Цена одной чашки капучино»:
    * Если в строке указан ценовой диапазон из двух значений, в столбец войдёт медиана этих двух значений.
    * Если в строке указано одно число — цена без диапазона, то в столбец войдёт это число.
    * Если значения нет или оно не начинается с подстроки «Цена одной чашки капучино», то в столбец ничего не войдёт.

#### Поля, важные для анализа
- category;
- district / category;
- chain / category;
- seats / category;
- rating / category;
- rating - корреляция с другими атрибутами;
- top 15 по name/avg rating/category;
- avg bill (?? или, по заданию middle_avg_bill) / district.

### Содержимое проекта
1. Загрузка данных и знакомство с ними.
2. Предобработка данных.
3. Исследовательский анализ данных.
4. Итоговый вывод и рекомендации.

## Конфигурирование среды

### Установка и импорты

In [1]:
!pip install phik
!pip install --upgrade matplotlib==3.4.3

In [2]:
import os
import sys
import numpy as np
import pandas as pd
from IPython.display import Markdown, HTML, display_html

# Графика, диаграммы и гистограммы, тепловые карты
import matplotlib
import matplotlib.ticker as mticker
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import seaborn as sns

# Расчёт коэффициента корреляции phi_k
import phik
from phik import phik_matrix

print(
    f"Путь к Python: {sys.executable}",
    f"Версия Python: {sys.version}",
    f"Текущий путь: {os.getcwd()}",
    f"pandas: {pd.__version__}",
    f"Бэкенд pandas (plotting): {pd.options.plotting.backend}",
    f"phik: {phik.__version__}",
    f"matplotlib: {matplotlib.__version__}",
    f"seaborn: {sns.__version__}",
sep='\n')

Путь к Python: /opt/conda/bin/python
Версия Python: 3.9.5 | packaged by conda-forge | (default, Jun 19 2021, 00:32:32) 
[GCC 9.3.0]
Текущий путь: /home/jovyan/work
pandas: 1.2.4
Бэкенд pandas (plotting): matplotlib
phik: 0.12.5
matplotlib: 3.4.3
seaborn: 0.11.1


 ### Общие функции

In [3]:
def show_ds_short_info(df):
    """
        Выводит информацию о размерности датасета и перечень колонок.
    """
    print(
        f'\tВсего: {df.shape[0]} строк, {df.shape[1]} колонок',
        f'\tКолонки датасета: {df.columns.tolist()}\n', sep='\n')
    
def show_item(num, name, action):
    """
        Отображает любой шаг анализа, заданный через func, в виде блока с заголовком.
    """
    print(f'{num}. {name}', '=' * 20)
    action()    
    
def show(start_num, *name_action):
    """
        Отображает шаги анализа с заголовком, нумерацией и разделением.
        Стартовая нумерация задайтся в start_num.
        Шаги переаются списком аргументов переменной длины: name, action. 
    """
    if len(name_action) % 2:
        raise ValueError('Нечётное число аргументов. В show передавайте пары значений name, action как единый список аргументов.')
    
    num_args = len(name_action)
    num = start_num
    
    for i in range(0, num_args, 2):
        if i != 0:
            print('\n')
        show_item(num, name_action[i], name_action[i + 1])
        num += 1
        if i < i - 2:
            print('\n')
            
def get_missing_values_statistics(df, name=None):
    """
        Проверяет все колонки DataFrame df на пропуски: np.nan, None, pd.NA.
        Возвращает DataFrame, содержащий две колоки: число пропусков и % пропусков по мнению функции isna( ).        
        В качестве индекса выступают имена колонок. Возвращаются только строки с ненулевыми данными.
    """
    subject = df if name is None else df[name]
    missing_count_seria = subject.isna().sum().loc[lambda v: v > 0].sort_values(ascending=False)
    missing_fraction = round(missing_count_seria / df.shape[0] * 100, 2)
    result = pd.DataFrame({'Пропущено (np.nan, None, pd.NA)': missing_count_seria, 'Доля пропусков %': missing_fraction})
    
    return 'В колонках DataFrame нет пропусков' if result.empty else result

def normalize_text_cols(df, *col_names, to_lower=True):
    """
        Конвертирует колонки col_names в string, обрезает пробелы с концов и нормализует пробелы внутри строк.
        Если задано to_lower, то приводит к нижнему регистру.
    """
    for name in col_names:        
        df[name] = df[name].astype('string').str.strip().str.replace(r'\s+', ' ', regex=True)
        if to_lower:
            df[name] = df[name].str.lower()
 
def get_whiskers(values, coeff=1.5, left=None, right=None):
    """
        Возвращает усы ящика для серии values.
    """
    Q1 = values.quantile(0.25)
    Q3 = values.quantile(0.75)
    IQR = Q3 - Q1
    
    w_left = Q1 - coeff * IQR
    w_right = Q3 + coeff * IQR
    
    if left is not None:
        w_left = max(left, w_left)
    if right is not None:
        w_right = min(right, w_right)
        
    return w_left, w_right

def get_reference_info(df, name, numeric=False):
    """
        Выполняет проверку серии name из DataFrame df как категории. А именно:
            - выводит TOP 150 уникальных значений серии и общее число уникальных значений;
            - выводит счётчик встречаемости и % для каждого значения в серии name;
            - если серия числовая, то ещё выводит min и max значения.
        Таким образом, помогает оценить качество справочника или просто серии (колонки) и заметить неявные дубликаты.
    """
    values = df[name].drop_duplicates().sort_values()
    lst = values.tolist()
    
    display(Markdown(f"**{name}:** `{lst[:150]}` = **{len(lst)}**\n\n"))
        
    counts_df = df[name].value_counts(dropna=False).astype('float64').to_frame('count')
    counts_df.index = counts_df.index.map(lambda x: '<NA>' if pd.isna(x) else x)
    counts_df['%'] = (counts_df['count'] / counts_df['count'].sum() * 100).round(2)        
    counts_df['count'] = counts_df['count'].astype('Int64')
             
    display(counts_df)
            
    print('\nПропущено:', df[name].isna().sum())
    
    if numeric:
        values_num = pd.to_numeric(df[name], errors='coerce')        
        min_val = values_num.min()
        max_val = values_num.max()
        print(f'min = {min_val}, mean = {round(values_num.mean(), 2)}, max = {max_val}, median = {values_num.median()}')
                
        whisker_left, whisker_right = get_whiskers(values_num, left=min_val, right=max_val)
        print(f'left_whisker = {whisker_left}, right_whisker = {whisker_right}')

        non_numeric = values[values_num.isna()].tolist()
        if non_numeric:
            print(f'Нецифровые значения: {non_numeric}')
    print('\n')
    
def get_frequency_report(df, name):
    """
        Возвращает частотный отчёт по колонке данных: все характеристики повторяемости значений в разных строках.
    """
    counts = df[name].value_counts(dropna=False)
    return pd.Series({
        'Среднее': counts.mean(),
        'Медиана': counts.median(), 
        'Мода': counts.mode().iloc[0] if len(counts.mode()) > 0 else None,
        'Макс': counts.max(),
        'Мин': counts.min()
    }), counts

def report_row_removing(before, after):
    print(f'✅ Из датасета в {before} строк удалено 🗑️ {before - after} строк,\n'
        f'\tосталось: {after} строк,\n'
        f'\tпроцент потерь: {round((before - after) / before * 100, 2)}%.')


## 1. Загрузка данных и знакомство с ними

- Загрузите данные о заведениях общественного питания Москвы. Путь к файлам: `/datasets/rest_info.csv` и `/datasets/rest_price.csv`.

### Конфигурация

In [4]:
BASE_DS_PATH = "https://code.s3.yandex.net/datasets/"
DATASET_INFO_PATH = BASE_DS_PATH + 'rest_info.csv'
DATASET_PRICE_PATH = BASE_DS_PATH + 'rest_price.csv'

### Загрузка датасетов

In [5]:
rest_info_df = pd.read_csv(DATASET_INFO_PATH, skipinitialspace=True)
rest_price_df = pd.read_csv(DATASET_PRICE_PATH, skipinitialspace=True)

# сохраним размерность исходных данных для сравнения после обработки
rest_info_df_shape = rest_info_df.shape
rest_price_df_shape = rest_price_df.shape

✅ Данные загружены, размерность сохранена на будущее.

- Познакомьтесь с данными и изучите общую информацию о них.

#### Датасет Точки питания rest_info.csv

In [6]:
show(1, 
     'Датасет "Точки питания"', lambda: show_ds_short_info(rest_info_df),
     'info', lambda: rest_info_df.info(),
     'Пропуски данных по колонкам', lambda: display(get_missing_values_statistics(rest_info_df)),
     'Пример данных', lambda: display(rest_info_df),
)

1. Датасет "Точки питания" ====================
	Всего: 8406 строк, 9 колонок
	Колонки датасета: ['id', 'name', 'category', 'address', 'district', 'hours', 'rating', 'chain', 'seats']



2. info ====================
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8406 entries, 0 to 8405
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   id        8406 non-null   object 
 1   name      8406 non-null   object 
 2   category  8406 non-null   object 
 3   address   8406 non-null   object 
 4   district  8406 non-null   object 
 5   hours     7870 non-null   object 
 6   rating    8406 non-null   float64
 7   chain     8406 non-null   int64  
 8   seats     4795 non-null   float64
dtypes: float64(2), int64(1), object(6)
memory usage: 591.2+ KB


3. Пропуски данных по колонкам ====================


,"Пропущено (np.nan, None, pd.NA)",Доля пропусков %
seats,3611,42.96
hours,536,6.38




4. Пример данных ====================


,id,name,category,address,district,hours,rating,chain,seats
0,0c3e3439a8c64ea5bf6ecd6ca6ae19f0,WoWфли,кафе,"Москва, улица Дыбенко, 7/1",Северный административный округ,"ежедневно, 10:00–22:00",5.0,0,NaN
1,045780ada3474c57a2112e505d74b633,Четыре комнаты,ресторан,"Москва, улица Дыбенко, 36, корп. 1",Северный административный округ,"ежедневно, 10:00–22:00",4.5,0,4.0
2,1070b6b59144425896c65889347fcff6,Хазри,кафе,"Москва, Клязьминская улица, 15",Северный административный округ,"пн-чт 11:00–02:00; пт,сб 11:00–05:00; вс 11:00...",4.6,0,45.0
3,03ac7cd772104f65b58b349dc59f03ee,Dormouse Coffee Shop,кофейня,"Москва, улица Маршала Федоренко, 12",Северный административный округ,"ежедневно, 09:00–22:00",5.0,0,NaN
4,a163aada139c4c7f87b0b1c0b466a50f,Иль Марко,пиццерия,"Москва, Правобережная улица, 1Б",Северный административный округ,"ежедневно, 10:00–22:00",5.0,1,148.0
...,...,...,...,...,...,...,...,...,...
8401,0342ad1a45ed41ba89dcba246a8267e5,Суши Мания,кафе,"Москва, Профсоюзная улица, 56",Юго-Западный административный округ,"ежедневно, 09:00–02:00",4.4,0,86.0
8402,ee6bb7c3650e47bd8186fca08eda1091,Миславнес,кафе,"Москва, Пролетарский проспект, 19, корп. 1",Южный административный округ,"ежедневно, 08:00–22:00",4.8,0,150.0
8403,62e8c64d4c89467aba608e39ef87616b,Самовар,кафе,"Москва, Люблинская улица, 112А, стр. 1",Юго-Восточный административный округ,"ежедневно, круглосуточно",3.9,0,150.0
8404,06a0db5ecd4842d48cd6350aa923e297,Чайхана Sabr,кафе,"Москва, Люблинская улица, 112А, стр. 1",Юго-Восточный административный округ,"ежедневно, круглосуточно",4.2,1,150.0


#### Датасет Цены rest_price.csv

In [7]:
show(1, 
     'Датасет "Цены"', lambda: show_ds_short_info(rest_price_df),
     'info', lambda: rest_price_df.info(),
     'Пропуски данных по колонкам', lambda: display(get_missing_values_statistics(rest_price_df)),
     'Пример данных', lambda: display(rest_price_df),
)

1. Датасет "Цены" ====================
	Всего: 4058 строк, 5 колонок
	Колонки датасета: ['id', 'price', 'avg_bill', 'middle_avg_bill', 'middle_coffee_cup']



2. info ====================
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4058 entries, 0 to 4057
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 4058 non-null   object 
 1   price              3315 non-null   object 
 2   avg_bill           3816 non-null   object 
 3   middle_avg_bill    3149 non-null   float64
 4   middle_coffee_cup  535 non-null    float64
dtypes: float64(2), object(3)
memory usage: 158.6+ KB


3. Пропуски данных по колонкам ====================


,"Пропущено (np.nan, None, pd.NA)",Доля пропусков %
middle_coffee_cup,3523,86.82
middle_avg_bill,909,22.40
price,743,18.31
avg_bill,242,5.96




4. Пример данных ====================


,id,price,avg_bill,middle_avg_bill,middle_coffee_cup
0,045780ada3474c57a2112e505d74b633,выше среднего,Средний счёт:1500–1600 ₽,1550.0,NaN
1,1070b6b59144425896c65889347fcff6,средние,Средний счёт:от 1000 ₽,1000.0,NaN
2,03ac7cd772104f65b58b349dc59f03ee,NaN,Цена чашки капучино:155–185 ₽,NaN,170.0
3,a163aada139c4c7f87b0b1c0b466a50f,средние,Средний счёт:400–600 ₽,500.0,NaN
4,8a343546b24e4a499ad96eb7d0797a8a,средние,NaN,NaN,NaN
...,...,...,...,...,...
4053,962800540173458486f3c465437c2d8b,средние,Цена бокала пива:от 140 ₽,NaN,NaN
4054,4277890d659341848d7051cbf2e51f51,выше среднего,Средний счёт:1000–1500 ₽,1250.0,NaN
4055,026cbc478f9b4c3294a83458dcd89982,средние,NaN,NaN,NaN
4056,71cc394927204d45b81c3a91edb47955,NaN,Средний счёт:50–250 ₽,150.0,NaN


#### Посмотрим пропуски, что в данных

In [8]:
missing1 = rest_info_df.query('seats.isna()')
missing2 = rest_info_df.query('seats == 0')
html1 = missing1[['name', 'category']].head(10).to_html()
html2 = missing2[['name', 'category']].head(10).to_html()

display_html(f'<div style="display: flex; gap: 20px;">{html1}{html2}</div>', raw=True)
print('Пропуски seats:', missing1.shape[0], '-',  round(missing1.shape[0] / rest_info_df_shape[0] * 100, 1), end='%; ')
print('Нули в seats:', missing2.shape[0], '-', round(missing2.shape[0] / rest_info_df_shape[0] * 100, 1), end='%')

rest_info_df.query('seats == 0')['category'].value_counts()

,name,category
0,WoWфли,кафе
3,Dormouse Coffee Shop,кофейня
5,Sergio Pizza,пиццерия
11,Шашлык Шефф,кафе
12,Заправка,кафе
14,У Сильвы,"бар,паб"
16,База Стритфуд,кафе
19,Пекарня,булочная
21,7/12,кафе
22,Крымские чебуреки,кафе


Пропуски seats: 3611 - 43.0%; Нули в seats: 136 - 1.6%

кафе               44
кофейня            24
ресторан           20
быстрое питание    18
булочная           11
пиццерия           10
столовая            5
бар,паб             4
Name: category, dtype: int64

In [9]:
print(rest_price_df['avg_bill'].str.split(':').str[0].astype('string').value_counts())

Средний счёт           3149
Цена чашки капучино     535
Цена бокала пива        132
Name: avg_bill, dtype: Int64


---

### Промежуточный вывод

Сделайте промежуточный вывод о полученных данных: данные какого объёма вам предоставили, соответствуют ли данные описанию, встречаются ли в них пропуски, используются ли верные типы данных. Отметьте другие особенности данных, которые вы обнаружите на этой стадии и на которые стоит обратить внимание при предобработке.

In [10]:
print(round(rest_price_df_shape[0] / rest_info_df_shape[0] * 100, 1), end='%')

48.3%

👉 В первом датасете **8406** строк, во втором **4058** строк. Колонки (9, 5) и их имена соотвествуют описанию и находятся в Snake Case. В обоих датасетах имеется ключевая колонка id, и согласно описанию второй датасет содержит данные о средних чеках, но **48% данных о чеках пропущено**. 

👉 Типы данных в датасетах универсальные, по-умолчанию. Требуют оптимизации. Сейчас это object, float64, int64. В строковых колонках тип Python, в числовых с пропусками откат к float из-за NaN. int только в chain, так как нет пропусков. Позже заменим типы данных, так как диапазоны будут ясны и типы тоже.

### Подготовка единого датафрейма

- Объедините данные двух датасетов в один, с которым вы и продолжите работу.

In [11]:
print(rest_info_df['id'].is_unique)
print(rest_price_df['id'].is_unique)

True
True


✅ Индексы по id уникальны. Пропусков нет. Установим их и приведём к string.

In [12]:
rest_info_df['id'] = rest_info_df['id'].astype('string')
rest_price_df['id'] = rest_price_df['id'].astype('string')

rest_info_df.set_index('id', inplace=True)
rest_price_df.set_index('id', inplace=True)

In [35]:
rest_df = rest_info_df.join(rest_price_df, how='left')
# сохраним shape для измерений
rest_df_shape = rest_df.shape
print(rest_df_shape)

(8406, 12)


✅ Единый датасет **rest_df**, как и ожидалось, 8 406 строк (все точки питания).

## 2. Предобработка данных

Подготовьте данные к исследовательскому анализу:

- Изучите корректность типов данных и при необходимости проведите их преобразование.

In [14]:
display(rest_df.dtypes)

name                  object
category              object
address               object
district              object
hours                 object
rating               float64
chain                  int64
seats                float64
price                 object
avg_bill              object
middle_avg_bill      float64
middle_coffee_cup    float64
dtype: object

👉 Типы данных не изменились. Теперь надо проверить катеригориальные и непрерывные значения в столбцах, чтобы подобрать типы данных.

### Проверим все строковые значения

In [15]:
display(rest_df.describe(include='object'))

,name,category,address,district,hours,price,avg_bill
count,8406,8406,8406,8406,7870,3315,3816
unique,5614,8,5753,9,1307,4,897
top,Кафе,кафе,"Москва, проспект Вернадского, 86В",Центральный административный округ,"ежедневно, 10:00–22:00",средние,Средний счёт:1000–1500 ₽
freq,189,2378,28,2242,759,2117,241


❗Видим, что 189 безымянных точек, больше всего точек типа Кафе, больше всего точек расположены в **ЦАО** и самый частый график работы **10-22**.

🆗 category без пропусков, больше всего **кафе**.

❗Есть странный адрес "москва, проспект вернадского, 86в", где зарегистрировано сразу 28 точек в одном месте. Проверить на дубликат.

🆗 district без пропусков, больше всего **ЦАО**.

❗hours есть пропуски 6%, надо анализировать чем заполнить. Рабочим будет новое поле **is_24_7**. Важно, чтобы оно было верным.

🆗 По ценовой категории price данные качественные, если не считать пропусков: самая популярная Средняя и счёт 1000-1500р.

👉 avg_bill надо разбить, выделив колонку-категорию avg_type.

### Проверим непрерывные значения

In [16]:
display(rest_df.describe())

,rating,chain,seats,middle_avg_bill,middle_coffee_cup
count,8406.000000,8406.000000,4795.000000,3149.000000,535.000000
mean,4.229895,0.381275,108.421689,958.053668,174.721495
std,0.470348,0.485729,122.833396,1009.732845,88.951103
min,1.000000,0.000000,0.000000,0.000000,60.000000
25%,4.100000,0.000000,40.000000,375.000000,124.500000
50%,4.300000,0.000000,75.000000,750.000000,169.000000
75%,4.400000,1.000000,140.000000,1250.000000,225.000000
max,5.000000,1.000000,1288.000000,35000.000000,1568.000000


❗Видим, что у seats std > mean и медиана 75, а среднее 108. min = 0. Значит, **пропуски 43%** могут быть неслучайными. Например, категория "Еда на вынос". Нужно обработать, проверить выбросы.
Добавить колонку 👉**has_seats**: 0, 1. seats нужен для анализа.

🆗 chain - данные хорошие. Пропусков нет.

❗rating - данные смещены в зону 4-4.5. Пропусков нет. Надо проверить на выбросы на графике. 📊

-----------

❗middle_avg_bill - проверить на выбросы 0 - 35 000. std больше среднего. 

❗middle_coffee_cup - можно слить в одну колонку **corrected_avg_bill** с middle_avg_bill в зависимости от avg_bill. Видимо зависит от категории точки. Проверить выбросы: 60 - 1568. Но, согласно задаче 8, там можно использовать middle_avg_bill. **Пока не будем.**❗

### Проверим катеригориальные значения

In [17]:
get_reference_info(rest_df, 'category', False)

**category:** `['бар,паб', 'булочная', 'быстрое питание', 'кафе', 'кофейня', 'пиццерия', 'ресторан', 'столовая']` = **8**



,count,%
кафе,2378,28.29
ресторан,2043,24.30
кофейня,1413,16.81
"бар,паб",765,9.10
пиццерия,633,7.53
быстрое питание,603,7.17
столовая,315,3.75
булочная,256,3.05



Пропущено: 0




🆗Category качественный. Без пропусков.

👉 Надо исследовать наличие мест и добавить поле has_seats = 0, 1.

In [18]:
get_reference_info(rest_df, 'district', False)

**district:** `['Восточный административный округ', 'Западный административный округ', 'Северный административный округ', 'Северо-Восточный административный округ', 'Северо-Западный административный округ', 'Центральный административный округ', 'Юго-Восточный административный округ', 'Юго-Западный административный округ', 'Южный административный округ']` = **9**



,count,%
Центральный административный округ,2242,26.67
Северный административный округ,900,10.71
Южный административный округ,892,10.61
Северо-Восточный административный округ,891,10.60
Западный административный округ,851,10.12
Восточный административный округ,798,9.49
Юго-Восточный административный округ,714,8.49
Юго-Западный административный округ,709,8.43
Северо-Западный административный округ,409,4.87



Пропущено: 0




🆗 District качественный. Без пропусков. 

👉 Для анализа нужна метрика удалённости от центра: distance: 0, 1, 2, где 0 - это центр, а 2 - это диагональ.

### Приведём числовые типы к оптимальным

✅ На основе анализа значений в колонках датасета можем подобрать типы:
- rating 1 - 5 (дробный): Float32; 🆗
- chain 0 - 1: Int8; 🆗
- seats 0 - 1288: Int16; ❗проверить на выброс

- middle_avg_bill 0 - 35 000: Int32; ❗проверить на выброс
- middle_avg_cap 60 - 1 568: Int32. ❗проверить на выброс

👉 Однако, с данными типами дружат не все виды преобразований и чартов seaborn, поэтому, иногда придётся преобразовывать налету.

In [19]:
rest_df = rest_df.astype({
    'rating': 'Float32', 'chain': 'Int8', 'seats': 'Int16',
    'middle_avg_bill': 'Int32', 'middle_coffee_cup': 'Int32',
    
    'name': 'string', 'category': 'string', 'address': 'string',
    'district': 'string', 'hours': 'string',
    'price': 'string', 'avg_bill': 'string'
})

display(rest_df.dtypes)

name                  string
category              string
address               string
district              string
hours                 string
rating               Float32
chain                   Int8
seats                  Int16
price                 string
avg_bill              string
middle_avg_bill        Int32
middle_coffee_cup      Int32
dtype: object

----------

- Изучите пропущенные значения в данных: посчитайте их количество в каждом столбце датафрейме, изучите данные с пропущенными значениями и предположите гипотезы их появления. Проведите обработку пропущенных значений: вы можете заменить пропуски на определённое значение, удалить строки с пропусками или оставить их как есть.

### Пропуски данных в колонках

Пропуски в колонках имеются в:
- **seats** - отсуствуют **43%, и ещё 1.6%** арифметические нули. По category видим, что это самые разные точки питания. Объективно, < 5 seats могут иметь ларьки и точки "на вынос", например, 1-2 стойки. Считаю, что это тип MAR. Считаю, что пропуски смешанные, часть из-за отсуствия данных, а часть из-за типа точки питания, но по category это точно не определить. Можно было бы заполнить пропуски медианами по category, но исключив нули и правые выбросы, но это не очень надёжно. Оставим, как есть.

👉 Для анализа добавим поле **seats_type**.


- **hours 6%**. hours нам нужны для поиска круглосуточных точек, харатер пропуска MCAR, объём умеренный, некритично. Запонить невозможно и не нужно.

В полях о среднем чеке видим, что есть некоторая степень бардака в данных. В задаче 8 просят анализировать цены на основе **middle_avg_bill**, он пропущен в **22% строк**. Однако, источником являются price и avg_bill. Из avg_bill мы берём цифру: среднее из двух или единичное значение, а по price, группировкой, можно построить синтетические данные и заполнить оставшиеся пропуски. avg_bill распарсен неполностью. Парсинг выполнен ошибочно: middle_avg_bill - это свалка из AVG и MIN значений. Не учтены пабы с ценой бокала пива, только кофейни с ценой чашки капучино и прочие заведения.

### Дубликаты

- Проверьте данные на явные и неявные дубликаты, например поля с названием и адресом заведения. Для оптимизации проверки нормализуйте данные в текстовых столбцах, например с названием заведения.

✅ Проверяем явные дубликаты, их нет.

In [20]:
print(rest_df.duplicated().sum())

0


✅ Текстовые строки по-умолчанию идут с типом object и могут иметь лишние пробелы внутри, а так же, набраны на разном регистре. Чтобы упростить их сравнение нормализуем их:
 - удаляем лишние пробелы;
 - приводим к lower case;
 - и к типу string, пропуски заменяем на pd.NA.

In [21]:
normalize_text_cols(rest_df, 'name', 'category', 'address', 'district', 'hours', 'price', 'avg_bill')

In [22]:
print(rest_df.duplicated().sum())

0


✅ Проверим и удалим явные дубликаты по полям name, address

In [34]:
alt_key = ['name', 'address']

# сортируем и добавляем длину строк в category и hours
rest_df = rest_df.assign(
    len_category = rest_df['category'].fillna('').str.len(),
    len_hours    = rest_df['hours'].fillna('').str.len(),
    total_len_cat_hours = rest_df['category'].fillna('').str.len() + rest_df['hours'].fillna('').str.len()
)

rest_df_sorted = (
    rest_df
    .sort_values(
        by=['name', 'address', 'total_len_cat_hours'],
        ascending=[True, True, False]
    )
)

# помечаем, какие строки вообще являются дубликатами по alt_key
is_duplicate = rest_df_sorted.duplicated(subset=alt_key, keep=False)

# сначала смотрим, какие дубли останутся
rest_df_dups_stay = (
    rest_df_sorted
    .loc[is_duplicate]
    .drop_duplicates(subset=alt_key, keep='first')
)

print("Дубликаты, которые останутся (с максимальной длиной category/hours):")
display(rest_df_dups_stay)

# определяем, какие дубли будут удалены
rest_df_dups_drop_indices = (
    rest_df_sorted
    .loc[is_duplicate]
    .drop_duplicates(subset=alt_key, keep='first')
    .index
)

rest_df_dups_removed = rest_df_sorted.loc[
    is_duplicate & (~rest_df_sorted.index.isin(rest_df_dups_drop_indices))
]

print("Дубликаты, которые будут удалены:")
display(rest_df_dups_removed)

# сама операция удаления (без вывода rest_df_clean)
rest_df_clean = rest_df_sorted.drop_duplicates(subset=alt_key, keep='first')
rest_df_clean = rest_df_clean.drop(columns=['len_category', 'len_hours', 'total_len_cat_hours'])

Дубликаты, которые останутся (с максимальной длиной category/hours):


,name,category,address,district,hours,rating,chain,seats,price,avg_bill,middle_avg_bill,middle_coffee_cup,len_category,len_hours,total_len_cat_hours
id,,,,,,,,,,,,,,,
a69f018d5c064873a3b491b0121bc1b4,more poke,ресторан,"москва, волоколамское шоссе, 11, стр. 2",северный административный округ,"пн-чт 09:00–18:00; пт,сб 09:00–21:00; вс 09:00...",4.2,1,188,<NA>,<NA>,<NA>,<NA>,8,52,60
072032ce16dc47bfbc63b672c75bd371,кафе,кафе,"москва, парк ангарские пруды",северный административный округ,"ежедневно, 09:00–23:00",3.2,0,<NA>,<NA>,<NA>,<NA>,<NA>,4,22,26
aba1de7ad7d64ac0a3f8684bda29d905,раковарня клешни и хвосты,"бар,паб","москва, проспект мира, 118",северо-восточный административный округ,"пн-чт 12:00–00:00; пт,сб 12:00–01:00; вс 12:00...",4.4,1,150,<NA>,<NA>,<NA>,<NA>,7,52,59
3c2a73ea79a04be48858fab3685f2f37,хлеб да выпечка,булочная,"москва, ярцевская улица, 19",западный административный округ,"ежедневно, 09:00–22:00",4.1,1,276,<NA>,<NA>,<NA>,<NA>,8,22,30


Дубликаты, которые будут удалены:


,name,category,address,district,hours,rating,chain,seats,price,avg_bill,middle_avg_bill,middle_coffee_cup,len_category,len_hours,total_len_cat_hours
id,,,,,,,,,,,,,,,
62608690e9cc464fbcd980cfd552e334,more poke,ресторан,"москва, волоколамское шоссе, 11, стр. 2",северный административный округ,"ежедневно, 09:00–21:00",4.2,0,188,<NA>,<NA>,<NA>,<NA>,8,22,30
897ddbc6746c4388b19dc8a9fcdbb488,кафе,кафе,"москва, парк ангарские пруды",северный административный округ,"ежедневно, 10:00–22:00",3.2,0,<NA>,<NA>,<NA>,<NA>,<NA>,4,22,26
c6ef39ae8a8c483d8f9a6531bc386a2c,раковарня клешни и хвосты,ресторан,"москва, проспект мира, 118",северо-восточный административный округ,"ежедневно, 12:00–00:00",4.4,0,150,<NA>,<NA>,<NA>,<NA>,8,22,30
d3116844e4e048f99614eb30be3214e0,хлеб да выпечка,кафе,"москва, ярцевская улица, 19",западный административный округ,<NA>,4.1,0,276,<NA>,<NA>,<NA>,<NA>,4,0,4


In [41]:
report_row_removing(rest_df_shape[0], rest_df_clean.shape[0])
rest_df = rest_df_clean

✅ Из датасета в 8406 строк удалено 🗑️ 4 строк,
	осталось: 8402 строк,
	процент потерь: 0.05%.


- Для дальнейшей работы создайте столбец `is_24_7` с обозначением того, что заведение работает ежедневно и круглосуточно, то есть 24/7:
  - логическое значение `True` — если заведение работает ежедневно и круглосуточно;
  - логическое значение `False` — в противоположном случае.

In [50]:
rest_df['is_24_7'] = rest_df['hours'].str.contains('ежедневно, круглосуточно', na=0).astype('Int8')

✅ Добавили поле для точек 24/7.

In [52]:
def district_distance(d):
    if pd.isna(d):
        return d
    if d.find('центральный') != -1:
        return 0
    if d.find('северо-') != -1 or d.find('юго-') != -1:
        return 2
    return 1
rest_df['distance'] = rest_df['district'].map(district_distance).astype('Int8')

✅ Добавили поле-метрику, для измерения расстояния от центра района расположения district.

---

### Промежуточный вывод

После предобработки данных напишите промежуточный вывод о проведённой работе. Отразите количество или долю отфильтрованных данных, если вы что-то удаляли.

In [54]:
print('Число строк ДО обработки:', rest_info_df_shape[0], '+', rest_price_df_shape[0], 
      ', ПОСЛЕ обработки:', rest_df.shape[0])

Число строк ДО обработки: 8406 + 4058 , ПОСЛЕ обработки: 8402


- После обработки у нас имеется 8402 записи о точках питания с уникальными id. Удалено **0.05%**❌ строк.


- Данные описанию соотвествуют и имеют верные имена колонок в Snake case.


- Пропуски есть в атрибуте seats ~43% и hours ~6%. Также, почти 50% пропусков в ценовом датасете, плюс в полях самого ценвого датасета, но внутри есть сильная кореляция полей, поэтому их можно пробовать восстанавливать.


- Типы данных **неоптимальные**, но неявных пропусков нет, поэтому их удалось привести к комактным Extended-типам pandas за один шаг.


- Атрибут **seats** нужен для анализа. Тут пропуски можно восстановить по среднему из группы category, district. Так же, число по category, если это "Еда на вынос"" или что-то в этом роде, где зала не предполагается. Для этого далее добавим флаг seats_type = 0, 1, 2 (нет мест, обычная ёмкость, большой зал).


- **middle_avg_bill** нужен для анализа в задаче. На мой взгляд, он не пригоден и его можно пересоздать и перезалить, заполнить пропуски цен во всех 4058 строках ценового датасета. Но, задание не просит. Оставим как есть.


- Добавлены колонки **is_24_7** и **distance**: для быстрого поиска точек 24/7 и для анализа зависимостей атрибутов от удалённости от центра.

## 3. Исследовательский анализ данных
Проведите исследовательский анализ исходных данных.

При исследовании данных используйте визуализации. Проверьте, что для каждого случая подобран оптимальный тип визуализации с корректным оформлением. У графика должен быть заголовок, понятные подписи по осям, при необходимости легенда, а его размер является оптимальным для изучения.

После исследования каждого пункта оставляйте небольшой комментарий с выводом или обсуждением результата. В конце шага обобщите результаты, выделив, по вашему мнению, самые важные.

---

### Задача 1

Какие категории заведений представлены в данных? Исследуйте количество объектов общественного питания по каждой категории. Результат сопроводите подходящей визуализацией.

---

### Задача 2

Какие административные районы Москвы присутствуют в данных? Исследуйте распределение количества заведений по административным районам Москвы, а также отдельно распределение заведений каждой категории в Центральном административном округе Москвы. Результат сопроводите подходящими визуализациями.

---

### Задача 3

Изучите соотношение сетевых и несетевых заведений в целом по всем данным и в разрезе категорий заведения. Каких заведений больше — сетевых или несетевых? Какие категории заведений чаще являются сетевыми? Исследуйте данные, ответьте на вопросы и постройте необходимые визуализации.

---

### Задача 4

Исследуйте количество посадочных мест в заведениях. Встречаются ли в данных аномальные значения или выбросы? Если да, то с чем они могут быть связаны? Приведите для каждой категории заведений наиболее типичное для него количество посадочных мест. Результат сопроводите подходящими визуализациями.


---

### Задача 5

Исследуйте рейтинг заведений. Визуализируйте распределение средних рейтингов по категориям заведений. Сильно ли различаются усреднённые рейтинги для разных типов общепита?

---

### Задача 6

Изучите, с какими данными показывают самую сильную корреляцию рейтинги заведений? Постройте и визуализируйте матрицу корреляции рейтинга заведения с разными данными: его категория, положение (административный район Москвы), статус сетевого заведения, количество мест, ценовая категория и признак, является ли заведения круглосуточным. Выберите самую сильную связь и проверьте её.

---

### Задача 7

Сгруппируйте данные по названиям заведений и найдите топ-15 популярных сетей в Москве. Для них посчитайте значения среднего рейтинга. Под популярностью понимается количество заведений этой сети в регионе. К какой категории заведений они относятся? Результат сопроводите подходящими визуализациями.

---

### Задача 8

Изучите вариацию среднего чека заведения (столбец `middle_avg_bill`) в зависимости от района Москвы. Проанализируйте цены в Центральном административном округе и других. Как удалённость от центра влияет на цены в заведениях? Результат сопроводите подходящими визуализациями.


---


---

### Промежуточный вывод

Обобщите полученные результаты, выделив, по вашему мнению, самые важные.

## 4. Итоговый вывод и рекомендации

По результатам проведённого исследовательского анализа данных сформулируйте итоговый вывод и рекомендации для заказчика. Старайтесь акцентировать внимание на ключевых моментах исследования.

При составлении вывода придерживайтесь такой структуры:

1. Общий обзор проделанной работы.
2. Ответы на исследовательские вопросы, или главные выводы.
3. Рекомендации на основе анализа данных.